In [11]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import leggauss
from scipy.linalg import solve
import sympy as sp

In [12]:
n=2
m=3
q=1
K=500
J = 20
epochs = 500
N_lambda = 5000
N = 10000
p2 = 1
p2_min = 1e-5
minroot = 1e-12
total_steps = 200000
T_intervall = (0,1)
t_0, t_end = T_intervall
N_c = 25
delta_T = (t_end-t_0)/N_c
M = 1000
OMEGA = (0, np.pi)
h = (OMEGA[1]-OMEGA[0])/M
eigen_values = []
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # Check for CUDA availability

In [13]:
# Define R(x) in parameterized form by using the following coefficients.
# a0,...aq are automatically determined through b0,..bq, by condition P2.
a2 = nn.Parameter(torch.zeros(n-q, device = device), requires_grad = True)
b1_log = nn.Parameter(torch.zeros(q, device = device), requires_grad = True)
b2_log =  nn.Parameter(torch.ones(m-q, device = device), requires_grad = True)
# Theta = (a2,b1_log,b2_log) are the parameters, that have to be optimized
# c are the coefficients of the taylor series of e^(-lambda) up to degree q
c_taylor = torch.tensor([(-1)**i / math.factorial(i) for i in range(q+1)], device = device)

def P(x):
  B = torch.cat((torch.ones(1, device = device), torch.exp(b1_log)))
  C_b = torch.zeros_like(B, device = device)
  for k in range(len(B)):
    for j in range(k+1):
      C_b[k] += c_taylor[j] * B[k - j]
  a = torch.cat((C_b, a2))
  p = torch.zeros_like(x, device = device)
  for coeff in a.flip(0):
    p = p * x +coeff
  return p

def Q(x):
  B = torch.cat((torch.ones(1, device = device), torch.exp(b1_log)))
  b = torch.cat((B, torch.exp(b2_log)))
  q = torch.zeros_like(x, device = device)
  for coeff in b.flip(0):
    q = q * x + coeff
  return q

def R(x):
  return P(x)/Q(x)




In [14]:

# two-stage Lobatto IIIC
def r2(x):
    return (2) / (x ** 2 + 2 * x + 2)

# three-stage Lobatto IIIC
def r3(x):
    return (-6 * x + 24) / (x ** 3 + 6 * x ** 2 + 18 * x + 24)

# four-stage Lobatto IIIC
def r4(x):
    return (12*x**2 - 120*x + 360) / (x**4 + 12*x**3 + 72*x**2 + 240*x + 360)

# three-stage Radau IIA
def rc(s):
    b = 0.5 * (1 + np.sqrt(3) / 3)
    return 1 - s / (1 + b * s) - np.sqrt(3) / 6 * (s / (1 + b * s))**2

def OCP_algorithm_div_c_grad_u(eigen_values,n,m,q,K,J,N,min_steps,epochs,N_lambda,p2,p2_min,convergence_threshold,minroot,total_steps,T_intervall,d):

  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # Check for CUDA availability


# We need to calculate the roots of the numerator of R'(x), we do this by calculating the coefficients symbolicaly,
# and later on calculate these coefficients, using the actual parameters from the current optimization step.

  x = sp.symbols("x")
  a_sym = sp.symbols(f"a0:{n+1}")
  b_sym = sp.symbols(f"b0:{m+1}")

  P_sym = sum(a_sym[i]*x**i for i in range(n+1))
  Q_sym = sum(b_sym[i]*x**i for i in range(m+1))

  # R'(x) = (P(x)/Q(x))' = (P'(x)*Q(x)-Q'(x)*P(x))/Q(x)^2, we only need the zeros of the numerator, therefore we
  # expand the numerator and then save the coefficients

  N_sym = sp.poly(sp.expand(sp.diff(P_sym, x)*Q_sym - sp.diff(Q_sym, x)*P_sym), x)

  #coefficients of the Numerator

  N_sym_coefficients = sp.lambdify((*a_sym,*b_sym), N_sym.all_coeffs(), "numpy")




#The set over which we optimize, in some experiments we used the exact eigenvalues:
  #lambdas = torch.tensor(delta_T * eigen_values, dtype = torch.float32, device = device)

#The way that was proposed in the paper uses the uniform Grid between the minimum and maximum Eigenvalue
  #lambdas = torch.linspace((delta_T * lambda_min),(delta_T * lambda_max), N_lambda)

#[0.01,100] was also used in a lot of examples and was also used for optimizing the stability function used for proposition 4.8
  lambdas = torch.linspace(0.01, 100, 5000, device=device)



  dataset = torch.utils.data.TensorDataset(lambdas)

  data_loader = torch.utils.data.DataLoader(dataset, batch_size=len(dataset))

  optimizer = torch.optim.SGD([b1_log, b2_log, a2], lr=1e-3)  #subgradient descent

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-15)

  best_params = None
  best_loss1 = float('inf')

  best_check_loss = float("inf")
  bad_checks = 0

  recent_losses = []

  for step in range(total_steps):
      optimizer.zero_grad()
      converged = False

      #L_s(Theta)
      loss1 = torch.max(torch.abs(r3(lambdas/J)**J - R(lambdas))/(1 - torch.abs(R(lambdas))))

      #L_b(Theta)
      #Second Loss, calculating the zeroz of R'(s), for the positive critical points of R(s)


      B = torch.cat((torch.ones(1, device = device), torch.exp(b1_log)))
      C_b = torch.zeros_like(B, device = device)
      for k in range(len(B)):
        for j in range(k+1):
          C_b[k] += c_taylor[j] * B[k - j]
      a = torch.cat((C_b, a2))

      b = torch.cat((B, torch.exp(b2_log)))

      coefficients = np.asarray(N_sym_coefficients(*a.detach().cpu().numpy(),*b.detach().cpu().numpy()), dtype=float).flatten()

      roots = np.roots(coefficients)

      positive_real_roots = roots[ (np.abs(roots.imag)<1e-8) & (roots.real > 1e-8)].real

      lambdas_barrier = torch.tensor(positive_real_roots, dtype = a2.dtype, device = a2.device)


      if n < m:

        if lambdas_barrier.size(0) > 0:
          loss2 = 1/(lambdas_barrier.size(0))*torch.sum(torch.log(torch.clamp(1 - R(lambdas_barrier)**2, min = minroot)))
        else:
          loss2 = torch.tensor(0.0, dtype = a2.dtype, device = a2.device)

      else:

        if lambdas_barrier.size(0) > 0:
          loss2 = 1/(lambdas_barrier.size(0))*torch.sum(torch.log(torch.clamp(1 - R(lambdas_barrier)**2, min = minroot))) + torch.log(torch.clamp(1 - (a2[-1] / torch.exp(b2_log[-1]))**2, min=minroot))
        else:
          loss2 = torch.log(torch.clamp(1 - (a2[-1] / torch.exp(b2_log[-1]))**2, min=minroot))



      if loss1.item() < best_loss1:
        best_params = (a2.clone().detach(), b1_log.clone().detach(), b2_log.clone().detach())
        best_loss1 = loss1.item()


      loss = loss1 - p2 * loss2
      loss.backward(retain_graph = True)
      optimizer.step()
      scheduler.step()

      if step % 100 == 0:
        p2 = max(p2 * 0.95, p2_min)    # coefficient infront of the berrier function in the loss gets decreased every 100 steps

      if step % 500 == 0:

        print(f" Iteration: {step}, Loss1: {loss1:.6f}, Loss2: {loss2:.6f}, Total Loss: {loss.item():.6f}")
        print("beste convergence rate = ", best_loss1)

      if step % 500 == 0:
        improvement = best_check_loss - best_loss1

        if best_loss1 < best_check_loss:
            best_check_loss = best_loss1

        if step >= min_steps:
            if improvement < convergence_threshold:
                bad_checks += 1
            else:
                bad_checks = 0

            if bad_checks >= patience:
                print(
                    f"stopped after patience at step {step}, "
                    f"best convergence factor = {best_loss1:.6f}, "
                    f"last improvement = {improvement:.2e}"
                )
                converged = True
                break



# Now we only need to take best_params and then give out the actual parameters as tensors a and b

  a2.data = best_params[0]
  b1_log.data = best_params[1]
  b2_log.data = best_params[2]

  A = torch.cat((torch.ones(1, device = device), torch.exp(b1_log)))
  C_b = torch.zeros_like(A, device = device)
  for k in range(len(A)):
    for j in range(k+1):
      C_b[k] += c_taylor[j] * A[k - j]
  a = torch.cat((C_b, a2))
  b = torch.cat((torch.ones(1, device = device), torch.exp(b1_log), torch.exp(b2_log)))

  return a, b, best_loss1

convergence_threshold = 1e-6
patience = 20

save_dir = "ocp_results"
os.makedirs(save_dir, exist_ok=True)

results = []

#d_values = [1,2,3,4,5,6,7,8,9,10,12,15] + list(range(20, 1000, 25))
d_values = [1]
results = []

for idx, d in enumerate(d_values):
    print(f"\nStarte Optimierung für d = {d}")

    if idx == 0:
        min_steps_current = 100000
    else:
        min_steps_current = 10000


    a, b, best_gamma = OCP_algorithm_div_c_grad_u(eigen_values,
        n, m, q, K, J, N, min_steps_current, epochs, N_lambda,
        p2, p2_min, convergence_threshold,
        minroot, total_steps, T_intervall,d
    )

    result = {
        "d": d,
        "min_steps": min_steps_current,
        "best_gamma": best_gamma,
        "a": a.detach().cpu(),
        "b": b.detach().cpu()
    }

    results.append(result)

    torch.save(results, os.path.join(save_dir, f"n{n}m{m}_001_100_LOBATTOIIIC3.pt"))
    print(f"Gespeichert für d = {d}, best_gamma = {best_gamma}")

# The results of the optimization, i.e. mainly the parameters are saved in a dictionary and can be downloaded.
# This is done so you can import them wherever you need them, without haven to run the optimization again, or without round off errors.
# The name of the saved file can be changed at the end. right now it contains the degrees n and m and the Set on which it was optimized,
# as well as the FP´s for which the OCP was optimized.




Starte Optimierung für d = 1
 Iteration: 0, Loss1: 0.347778, Loss2: 0.000000, Total Loss: 0.347778
beste convergence rate =  0.34777840971946716
 Iteration: 500, Loss1: 0.322717, Loss2: 0.000000, Total Loss: 0.322717
beste convergence rate =  0.32271742820739746
 Iteration: 1000, Loss1: 0.292609, Loss2: 0.000000, Total Loss: 0.292609
beste convergence rate =  0.29260873794555664
 Iteration: 1500, Loss1: 0.255583, Loss2: 0.000000, Total Loss: 0.255583
beste convergence rate =  0.2555826008319855
 Iteration: 2000, Loss1: 0.208515, Loss2: 0.000000, Total Loss: 0.208515
beste convergence rate =  0.208515465259552
 Iteration: 2500, Loss1: 0.145808, Loss2: 0.000000, Total Loss: 0.145808
beste convergence rate =  0.14580754935741425
 Iteration: 3000, Loss1: 0.056734, Loss2: 0.000000, Total Loss: 0.056734
beste convergence rate =  0.056734055280685425
 Iteration: 3500, Loss1: 0.052198, Loss2: 0.000000, Total Loss: 0.052198
beste convergence rate =  0.05218945071101189
 Iteration: 4000, Loss1: